# Rodar pipelines de habitats

Edite **somente** a célula de configuração abaixo e depois execute todas as células em ordem.
Os arquivos `Habitats.py`, `Features.py` e `Metricas.py` devem estar na mesma pasta deste notebook.

In [ ]:
import time
import os
import numpy as np
import Habitats
import Metricas

## Configuração

- `pasta_casos`, `pasta_saida`, `pasta_visualizacoes`: caminhos das pastas (relativos a este notebook ou absolutos).
- `imagens`: um item por caso, com o caminho da imagem e da máscara. Adicione ou remova itens conforme o número de casos. Todos os casos são clusterizados juntos, então a **ordem** desta lista define a ordem dos casos em todas as etapas seguintes.
- `configuracoes`: uma execução por linha `(sigma, kernel, raio_fechamento, passo)`. A **primeira linha** é usada como referência nas métricas.

In [ ]:
# ======================== EDITE AQUI ========================

pasta_casos = "Casos"
pasta_saida = "Saidas"
pasta_visualizacoes = "Visualizacoes"

# Cada item: nome (usado nas mensagens), caminho da imagem e caminho da máscara
imagens = [
    {
        "nome": "caso_1",
        "imagem": os.path.join(pasta_casos, "exame_1.mha"),
        "mascara": os.path.join(pasta_casos, "mascara_1.mha"),
    },
    {
        "nome": "caso_2",
        "imagem": os.path.join(pasta_casos, "exame_2.mha"),
        "mascara": os.path.join(pasta_casos, "mascara_2.mha"),
    },
]

configuracoes = [
    # (sigma, kernel, raio_fechamento, passo)
    (0.4, 3, 1, 1),   # REFERÊNCIA (sempre a primeira linha)
    (0.2, 3, 1, 1),   # sigma
    (0.8, 3, 1, 1),
    (1,   3, 1, 1),
    (0.4, 5, 1, 1),   # kernel
    (0.4, 7, 1, 1),
    (0.4, 9, 1, 1),
    (0.4, 3, 2, 1),   # raio de fechamento
    (0.4, 3, 3, 1),
    (0.4, 3, 4, 1),
    (0.4, 3, 1, 2),   # passo
    (0.4, 3, 1, 3),
    (0.4, 3, 1, 4),
]

# ============================================================

caminhos_imagens = [item["imagem"] for item in imagens]
caminhos_mascaras = [item["mascara"] for item in imagens]

# Interrompe logo no início se algum arquivo não for encontrado.
faltando = [c for c in caminhos_imagens + caminhos_mascaras if not os.path.isfile(c)]
if faltando:
    raise FileNotFoundError("Arquivos não encontrados:\n" + "\n".join(faltando))

## Pipeline de segmentação por grid regular (todos os casos clusterizados juntos)

Uma execução por configuração de parâmetros. Cada execução grava em `pasta_saida` os rótulos de cada caso, os centroides e os parâmetros usados.

In [ ]:
def rodar_configuracao(sigma, kernel, raio_fechamento, passo):
    """Roda o pipeline pooled para uma configuração.

    Grava os rótulos, centroides e parâmetros em pasta_saida/<config>, e as
    figuras dos habitats em pasta_visualizacoes/<config>, um PDF por caso.
    Nenhuma figura é exibida na saída da célula.
    """
    nome = f"s{sigma}_k{kernel}_r{raio_fechamento}_p{passo}"
    print(f"Habitats iniciado — sigma={sigma}, kernel={kernel}, raio={raio_fechamento}, passo={passo}")
    tempo_inicio = time.perf_counter()
    Habitats.pipeline_habitats_GridRegular_Pooled(
        caminhos_imagens,
        caminhos_mascaras,
        sigma=sigma,
        kernel=kernel,
        raio_fechamento=raio_fechamento,
        passo=passo,
        caminho_saida=os.path.join(pasta_saida, nome),
        pasta_visualizacoes=pasta_visualizacoes,
    )
    tempo_execucao_s = time.perf_counter() - tempo_inicio
    print(f"Tempo de processamento (segmentação + extração + clusterização): {tempo_execucao_s:.1f} s")

In [ ]:
for sigma, kernel, raio_fechamento, passo in configuracoes:
    rodar_configuracao(sigma, kernel, raio_fechamento, passo)

In [ ]:
execucoes = Metricas.carregar_execucoes(pasta_saida)
if execucoes["_ignoradas"]:
    print("ATENÇÃO — execuções ignoradas (arquivos faltando):", execucoes["_ignoradas"])

# A referência é a primeira linha de `configuracoes`.
parametros_referencia = dict(zip(("sigma", "kernel", "raio_fechamento", "passo"), configuracoes[0]))
id_referencia = Metricas.identificar_referencia(execucoes, parametros_ref=parametros_referencia)
referencia = execucoes[id_referencia]

mascaras = [
    Habitats.ler_caso_isotropico(img, msk)[1]
    for img, msk in zip(caminhos_imagens, caminhos_mascaras)
]

tabelas = []
for id_exec, teste in execucoes.items():
    if id_exec == "_ignoradas":
        continue

    tabela = Metricas.tabela_estabilidade_execucao(
        referencia["volumes"],
        teste["volumes"],
        mascaras,
        referencia["centros"],
        teste["centros"],
        metadados={"id_execucao": id_exec, **teste["parametros"]},
    )
    tabelas.append(tabela)

resultados = Metricas.consolidar_tabelas(
    tabelas, caminho_csv=os.path.join(pasta_saida, "metricas_estabilidade.csv")
)
resultados